# SearxNG 搜索

本笔记本将介绍如何使用自托管的 `SearxNG` 搜索 API 来搜索网络。

您可以访问此链接以获取有关 `Searx API` 参数的更多信息：[https://docs.searxng.org/dev/search_api.html](https://docs.searxng.org/dev/search_api.html)

In [ ]:
import pprint

from langchain_community.utilities import SearxSearchWrapper

In [ ]:
search = SearxSearchWrapper(searx_host="http://127.0.0.1:8888")

对于某些引擎，如果提供了直接的`answer`，包装器将打印该答案而不是完整的搜索结果列表。如果您想获取所有结果，可以使用包装器的`results`方法。

In [1]:
search.run("What is the capital of France")

'Paris is the capital of France, the largest country of Europe with 550 000 km2 (65 millions inhabitants). Paris has 2.234 million inhabitants end 2011. She is the core of Ile de France region (12 million people).'

## 自定义参数

SearxNG 支持 [135 个搜索引擎](https://docs.searxng.org/user/configured_engines.html)。您还可以使用任意命名的参数来自定义 Searx 包装器，这些参数将传递给 Searx 搜索 API。在下面的示例中，我们将对来自 searx 搜索 API 的自定义搜索参数进行更有趣的运用。

在此示例中，我们将使用 `engines` 参数来查询维基百科

In [ ]:
search = SearxSearchWrapper(
    searx_host="http://127.0.0.1:8888", k=5
)  # k is for max number of items

In [2]:
search.run("large language model ", engines=["wiki"])

'Large language models (LLMs) represent a major advancement in AI, with the promise of transforming domains through learned knowledge. LLM sizes have been increasing 10X every year for the last few years, and as these models grow in complexity and size, so do their capabilities.\n\nGPT-3 can translate language, write essays, generate computer code, and more — all with limited to no supervision. In July 2020, OpenAI unveiled GPT-3, a language model that was easily the largest known at the time. Put simply, GPT-3 is trained to predict the next word in a sentence, much like how a text message autocomplete feature works.\n\nA large language model, or LLM, is a deep learning algorithm that can recognize, summarize, translate, predict and generate text and other content based on knowledge gained from massive datasets. Large language models are among the most successful applications of transformer models.\n\nAll of today’s well-known language models—e.g., GPT-3 from OpenAI, PaLM or LaMDA from

为 searx 传递其他 Searx 参数，例如 `language`

In [3]:
search = SearxSearchWrapper(searx_host="http://127.0.0.1:8888", k=1)
search.run("deep learning", language="es", engines=["wiki"])

'Aprendizaje profundo (en inglés, deep learning) es un conjunto de algoritmos de aprendizaje automático (en inglés, machine learning) que intenta modelar abstracciones de alto nivel en datos usando arquitecturas computacionales que admiten transformaciones no lineales múltiples e iterativas de datos expresados en forma matricial o tensorial. 1'

## 获取带元数据的结果

You can retrieve results along with their associated metadata. This is useful for understanding the context of your results or for debugging.

您可以检索结果及其关联的元数据。这对于理解结果的上下文或进行调试非常有用。

```jsx
import { useQuery } from '@tanstack/react-query';
import axios from 'axios';

function Example() {
  const { data, error, isLoading, isFetching, isSuccess } = useQuery({
    queryKey: ['todos'],
    queryFn: async () => {
      const response = await axios.get('/api/todos');
      return response.data;
    },
  });

  if (isLoading) {
    return <span>Loading...</span>;
  }

  if (error) {
    return <span>Error: {error.message}</span>;
  }

  return (
    <div>
      <h1>Todos</h1>
      <ul>
        {data.map((todo) => (
          <li key={todo.id}>{todo.title}</li>
        ))}
      </ul>
      {/* You can access metadata here */}
      {isFetching && <p>Refreshing data...</p>}
      {isSuccess && <p>Data fetched successfully!</p>}
    </div>
  );
}
```

In this example:

在此示例中：

*   `isLoading`: A boolean that is `true` while the query is executing for the first time.
    *   `isLoading`：一个布尔值，在查询首次执行时为 `true`。
*   `isFetching`: A boolean that is `true` whenever the query is fetching data, regardless of whether it's the initial fetch or a background refetch.
    *   `isFetching`：一个布尔值，只要查询正在获取数据，它就为 `true`，无论这是初始获取还是后台重新获取。
*   `isSuccess`: A boolean that is `true` if the query has successfully fetched data at least once.
    *   `isSuccess`：一个布尔值，如果查询至少成功获取一次数据，则为 `true`。
*   `error`: An object containing error information if the query failed.
    *   `error`：如果查询失败，则包含错误信息的对象。

You can use these metadata flags to provide feedback to your users about the state of the data fetching process.

您可以使用这些元数据标志向用户提供有关数据获取过程状态的反馈。

在此示例中，我们将使用 `categories` 参数查找科学论文，并将结果限制在 `time_range` 内（并非所有引擎都支持时间范围选项）。

我们还希望以结构化的方式获取包含元数据的结果。为此，我们将使用包装器的 `results` 方法。

In [ ]:
search = SearxSearchWrapper(searx_host="http://127.0.0.1:8888")

In [4]:
results = search.results(
    "Large Language Model prompt",
    num_results=5,
    categories="science",
    time_range="year",
)
pprint.pp(results)

[{'snippet': '… on natural language instructions, large language models (… the '
             'prompt used to steer the model, and most effective prompts … to '
             'prompt engineering, we propose Automatic Prompt …',
  'title': 'Large language models are human-level prompt engineers',
  'link': 'https://arxiv.org/abs/2211.01910',
  'engines': ['google scholar'],
  'category': 'science'},
 {'snippet': '… Large language models (LLMs) have introduced new possibilities '
             'for prototyping with AI [18]. Pre-trained on a large amount of '
             'text data, models … language instructions called prompts. …',
  'title': 'Promptchainer: Chaining large language model prompts through '
           'visual programming',
  'link': 'https://dl.acm.org/doi/abs/10.1145/3491101.3519729',
  'engines': ['google scholar'],
  'category': 'science'},
 {'snippet': '… can introspect the large prompt model. We derive the view '
             'ϕ0(X) and the model h0 from T01. However, 

从 arXiv 获取论文

In [5]:
results = search.results(
    "Large Language Model prompt", num_results=5, engines=["arxiv"]
)
pprint.pp(results)

[{'snippet': 'Thanks to the advanced improvement of large pre-trained language '
             'models, prompt-based fine-tuning is shown to be effective on a '
             'variety of downstream tasks. Though many prompting methods have '
             'been investigated, it remains unknown which type of prompts are '
             'the most effective among three types of prompts (i.e., '
             'human-designed prompts, schema prompts and null prompts). In '
             'this work, we empirically compare the three types of prompts '
             'under both few-shot and fully-supervised settings. Our '
             'experimental results show that schema prompts are the most '
             'effective in general. Besides, the performance gaps tend to '
             'diminish when the scale of training data grows large.',
  'title': 'Do Prompts Solve NLP Tasks Using Natural Language?',
  'link': 'http://arxiv.org/abs/2203.00902v1',
  'engines': ['arxiv'],
  'category': 'science'},
 

在此示例中，我们在 `it` 类别下查询 `large language models`。然后，我们过滤来自 GitHub 的结果。

In [6]:
results = search.results("large language model", num_results=20, categories="it")
pprint.pp(list(filter(lambda r: r["engines"][0] == "github", results)))

[{'snippet': 'Guide to using pre-trained large language models of source code',
  'title': 'Code-LMs',
  'link': 'https://github.com/VHellendoorn/Code-LMs',
  'engines': ['github'],
  'category': 'it'},
 {'snippet': 'Dramatron uses large language models to generate coherent '
             'scripts and screenplays.',
  'title': 'dramatron',
  'link': 'https://github.com/deepmind/dramatron',
  'engines': ['github'],
  'category': 'it'}]


我们也可以直接从 `github` 和其他源代码托管平台查询结果。

In [7]:
results = search.results(
    "large language model", num_results=20, engines=["github", "gitlab"]
)
pprint.pp(results)

[{'snippet': "Implementation of 'A Watermark for Large Language Models' paper "
             'by Kirchenbauer & Geiping et. al.',
  'title': 'Peutlefaire / LMWatermark',
  'link': 'https://gitlab.com/BrianPulfer/LMWatermark',
  'engines': ['gitlab'],
  'category': 'it'},
 {'snippet': 'Guide to using pre-trained large language models of source code',
  'title': 'Code-LMs',
  'link': 'https://github.com/VHellendoorn/Code-LMs',
  'engines': ['github'],
  'category': 'it'},
 {'snippet': '',
  'title': 'Simen Burud / Large-scale Language Models for Conversational '
           'Speech Recognition',
  'link': 'https://gitlab.com/BrianPulfer',
  'engines': ['gitlab'],
  'category': 'it'},
 {'snippet': 'Dramatron uses large language models to generate coherent '
             'scripts and screenplays.',
  'title': 'dramatron',
  'link': 'https://github.com/deepmind/dramatron',
  'engines': ['github'],
  'category': 'it'},
 {'snippet': 'Code for loralib, an implementation of "LoRA: Low-Rank '
   